In [1]:
!uv pip install mantis-tsfm catboost lightgbm

Using Python 3.13.2 environment at: /home/petelin/TSCGlue/.venv
Audited 3 packages in 15ms


In [2]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import accuracy_score
from mantis.architecture import MantisV2
from mantis.trainer import MantisTrainer
from tscglue import data_loader

In [3]:
dataset = "Worms"
dataset = 'Car'
# dataset = 'HandOutlines'
# dataset = 'Trace'
dataset = 'SwedishLeaf'
# dataset = 'FordA'
X_train, y_train, X_test, y_test = data_loader.load_fold(dataset, 0)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(500, 1, 128) (500,) (625, 1, 128) (625,)


In [4]:
from tscglue.models_tsfm import ALL_TSFM_MODELS, make_tsfm_model
from time import perf_counter

for model_name in ALL_TSFM_MODELS:
    t0 = perf_counter()
    m = make_tsfm_model(model_name)
    m.fit(X_train, y_train)
    y_pred = m.predict(X_test)
    elapsed = perf_counter() - t0
    print(f"{model_name}: {accuracy_score(y_test, y_pred):.4f} ({elapsed:.1f}s)")

2026-03-10 17:13:30.102763: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


mantis-ridgecv: 0.9536 (5.9s)
mantis-rf: 0.9264 (5.9s)
mantis-et: 0.9264 (5.4s)
mantis-hgb: 0.9312 (11.9s)


/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


mantis-lgbm: 0.9280 (7.6s)
chronos2-ridgecv: 0.9264 (16.6s)
chronos2-rf: 0.8400 (30.0s)
chronos2-et: 0.8480 (18.2s)
chronos2-hgb: 0.8704 (127.6s)


/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


chronos2-lgbm: 0.8624 (332.6s)
mantis+chronos2-ridgecv: 0.9600 (36.5s)
mantis+chronos2-rf: 0.9168 (23.7s)
mantis+chronos2-et: 0.9328 (20.5s)
mantis+chronos2-hgb: 0.9344 (48.8s)
mantis+chronos2-lgbm: 0.9248 (23.6s)


/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [5]:
device = 'cpu'
network = MantisV2(device=device)
network = network.from_pretrained("paris-noah/MantisV2")
model = MantisTrainer(device=device, network=network)


In [6]:
def resize(X, size=512):
    return F.interpolate(
        torch.tensor(X, dtype=torch.float), size=size, mode='linear', align_corners=False
    ).numpy()

X_train_r = resize(X_train)
X_test_r = resize(X_test)
print(X_train_r.shape, X_test_r.shape)

(500, 1, 512) (625, 1, 512)


In [7]:
# Extract frozen embeddings (no fine-tuning)
Z_train = model.transform(X_train_r)
Z_test = model.transform(X_test_r)
print(Z_train.shape, Z_test.shape)

(500, 256) (625, 256)


In [8]:
# Classify on frozen embeddings
from sklearn.linear_model import RidgeClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

pipe = make_pipeline(StandardScaler(), RidgeClassifierCV(alphas=np.logspace(-3, 3, 10)))
pipe.fit(Z_train, y_train)
y_pred = pipe.predict(Z_test)
print(f"Mantis embeddings + RidgeCV accuracy: {accuracy_score(y_test, y_pred):.4f}")

Mantis embeddings + RidgeCV accuracy: 0.9536


In [9]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

for name, clf in [
    ("RidgeCV", RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))),
    ("RandomForest", RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)),
    ("ExtraTrees", ExtraTreesClassifier(n_estimators=500, random_state=42, n_jobs=-1)),
    ("HistGradientBoosting", HistGradientBoostingClassifier(max_iter=500, random_state=42)),
    ("LightGBM", LGBMClassifier(n_estimators=500, random_state=42, verbose=-1)),
    ("CatBoost", CatBoostClassifier(iterations=500, random_seed=42, verbose=0)),
]:
    pipe = make_pipeline(StandardScaler(), clf)
    pipe.fit(Z_train, y_train)
    y_pred = pipe.predict(Z_test)
    print(f"Mantis embeddings + {name}: {accuracy_score(y_test, y_pred):.4f}")

Mantis embeddings + RidgeCV: 0.9536
Mantis embeddings + RandomForest: 0.9264
Mantis embeddings + ExtraTrees: 0.9264
Mantis embeddings + HistGradientBoosting: 0.9312


/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Mantis embeddings + LightGBM: 0.9280
Mantis embeddings + CatBoost: 0.9264


In [10]:
Z_train.shape

(500, 256)

In [11]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

model.fit(X_train_r, y_train_enc, num_epochs=1)
y_pred = le.inverse_transform(model.predict(X_test_r))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

Epoch 0: Train Loss 2.7676: 100%|██████████| 1/1 [00:06<00:00,  6.02s/it]


Accuracy: 0.1344


In [12]:
from chronos import BaseChronosPipeline

chronos_pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cpu")

In [13]:
def chronos_embed(pipeline, X, batch_size=64):
    all_embs = []
    for i in range(0, len(X), batch_size):
        batch = [torch.from_numpy(x.squeeze(0)).float() for x in X[i:i+batch_size]]
        embeddings, _ = pipeline.embed(batch)
        # [REG] token at position [-2] per series
        vecs = [e[0, -2, :].detach().cpu().numpy() for e in embeddings]
        all_embs.append(np.stack(vecs))
    return np.vstack(all_embs)

Z_train_c = chronos_embed(chronos_pipeline, X_train)
Z_test_c = chronos_embed(chronos_pipeline, X_test)
print(Z_train_c.shape, Z_test_c.shape)

(500, 768) (625, 768)


In [14]:
for name, clf in [
    ("RidgeCV", RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))),
    ("RandomForest", RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)),
    ("ExtraTrees", ExtraTreesClassifier(n_estimators=500, random_state=42, n_jobs=-1)),
    ("HistGradientBoosting", HistGradientBoostingClassifier(max_iter=500, random_state=42)),
    ("LightGBM", LGBMClassifier(n_estimators=500, random_state=42, verbose=-1)),
    #("CatBoost", CatBoostClassifier(iterations=500, random_seed=42, verbose=0)),
]:
    pipe = make_pipeline(StandardScaler(), clf)
    
    pipe.fit(Z_train_c, y_train)
    y_pred = pipe.predict(Z_test_c)
    print(f"Chronos2 embeddings + {name}: {accuracy_score(y_test, y_pred):.4f}")

Chronos2 embeddings + RidgeCV: 0.9264
Chronos2 embeddings + RandomForest: 0.8400
Chronos2 embeddings + ExtraTrees: 0.8480
Chronos2 embeddings + HistGradientBoosting: 0.8704
Chronos2 embeddings + LightGBM: 0.8624


/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [15]:
# Combine Mantis (256d) + Chronos2 (768d) embeddings
Z_train_combined = np.hstack([Z_train, Z_train_c])
Z_test_combined = np.hstack([Z_test, Z_test_c])
print(Z_train_combined.shape, Z_test_combined.shape)

for name, clf in [
    ("RidgeCV", RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))),
    ("RandomForest", RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)),
    ("ExtraTrees", ExtraTreesClassifier(n_estimators=500, random_state=42, n_jobs=-1)),
    ("HistGradientBoosting", HistGradientBoostingClassifier(max_iter=500, random_state=42)),
    ("LightGBM", LGBMClassifier(n_estimators=500, random_state=42, verbose=-1)),
    #("CatBoost", CatBoostClassifier(iterations=500, random_seed=42, verbose=0)),
]:
    pipe = make_pipeline(StandardScaler(), clf)
    pipe.fit(Z_train_combined, y_train)
    y_pred = pipe.predict(Z_test_combined)
    print(f"Mantis+Chronos2 embeddings + {name}: {accuracy_score(y_test, y_pred):.4f}")

(500, 1024) (625, 1024)
Mantis+Chronos2 embeddings + RidgeCV: 0.9600
Mantis+Chronos2 embeddings + RandomForest: 0.9168
Mantis+Chronos2 embeddings + ExtraTrees: 0.9328
Mantis+Chronos2 embeddings + HistGradientBoosting: 0.9344
Mantis+Chronos2 embeddings + LightGBM: 0.9248


/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [16]:
from tscglue.models_tsfm import ALL_TSFM_MODELS, make_tsfm_model
from time import perf_counter

for model_name in ALL_TSFM_MODELS:
    t0 = perf_counter()
    m = make_tsfm_model(model_name)
    m.fit(X_train, y_train)
    y_pred = m.predict(X_test)
    elapsed = perf_counter() - t0
    print(f"{model_name}: {accuracy_score(y_test, y_pred):.4f} ({elapsed:.1f}s)")

mantis-ridgecv: 0.9536 (3.1s)
mantis-rf: 0.9264 (4.8s)
mantis-et: 0.9264 (6.2s)
mantis-hgb: 0.9312 (58.1s)


/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


mantis-lgbm: 0.9280 (8.7s)
chronos2-ridgecv: 0.9264 (19.9s)
chronos2-rf: 0.8400 (27.1s)
chronos2-et: 0.8480 (66.8s)
chronos2-hgb: 0.8704 (150.0s)


/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


chronos2-lgbm: 0.8624 (28.4s)
mantis+chronos2-ridgecv: 0.9600 (19.3s)
mantis+chronos2-rf: 0.9168 (32.1s)
mantis+chronos2-et: 0.9328 (28.4s)
mantis+chronos2-hgb: 0.9344 (30.1s)
mantis+chronos2-lgbm: 0.9248 (25.5s)


/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [3]:
import numpy as np
X = np.random.rand(100, 50)

In [9]:
! uv pip install tabicl

from tabicl import TabICLClassifier
clf = TabICLClassifier()
clf.fit(X, np.random.randint(0, 2, size=100))
y_pred = clf.predict(X)


Using Python 3.13.2 environment at: /home/petelin/TSCGlue/.venv
Audited 1 package in 11ms
